# Tối ưu hoá đọc dữ liệu trong Spark — Notebook thực hành

Notebook này giúp bạn **thực sự cảm nhận** sự khác biệt giữa các kỹ thuật tối ưu đọc dữ liệu thông qua đo lường cụ thể (thời gian, lượng I/O, kế hoạch thực thi).

## Nội dung

1. **Setup**: tạo SparkSession và sinh dữ liệu mẫu (~1 triệu dòng, 20 cột)
2. **CSV vs Parquet**: so sánh kích thước file và tốc độ đọc
3. **Predicate Pushdown**: đo lường filter trên Parquet vs CSV
4. **Column Pruning**: so sánh `select` 2 cột vs đọc toàn bộ
5. **Partition Pruning**: ghi với `partitionBy` và đo I/O
6. **Kết hợp cả 3**: query thực tế trên data lake mô phỏng
7. **Internal vs External Table**: thực nghiệm `DROP TABLE`

## Yêu cầu

```bash
pip install pyspark==3.5.0
```

Notebook chạy được trên máy local — không cần cluster.

---
## 1. Setup — SparkSession và dữ liệu mẫu

In [1]:
import os
import time
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Khởi tạo SparkSession
spark = (SparkSession.builder
         .appName("ReadOptimization")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")  # giảm cho local
         .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 13:02:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8


In [2]:
# Sinh dữ liệu mẫu: 100 dòng, 20 cột, mô phỏng bảng sales
# - year (2023, 2024, 2025), month (1-12)
# - city (5 thành phố), category (10 loại)
# - 16 cột số phụ để mô phỏng bảng nhieu cot — testing column pruning

N_ROWS = 1000000

df = (spark.range(0, N_ROWS)
      .withColumn("year",     (F.col("id") % 3 + 2023).cast("int"))
      .withColumn("month",    (F.col("id") % 12 + 1).cast("int"))
      .withColumn("city",     F.element_at(F.array(
          F.lit("HCM"), F.lit("HN"), F.lit("DN"), F.lit("CT"), F.lit("HP")
      ), (F.col("id") % 5 + 1).cast("int")))
      .withColumn("category", F.concat(F.lit("cat_"), (F.col("id") % 10).cast("string")))
      .withColumn("amount",   (F.rand() * 1000).cast("double"))
      .withColumn("quantity", (F.col("id") % 100).cast("int")))

# Thêm 14 cột số phụ để bảng nhieu cot hơn — quan trọng cho demo column pruning
for i in range(14):
    df = df.withColumn(f"extra_col_{i}", (F.rand() * 10000).cast("double"))

print(f"Tổng số cột: {len(df.columns)}")
df.printSchema()
df.show(3, truncate=False)

Tổng số cột: 21
root
 |-- id: long (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- extra_col_0: double (nullable = false)
 |-- extra_col_1: double (nullable = false)
 |-- extra_col_2: double (nullable = false)
 |-- extra_col_3: double (nullable = false)
 |-- extra_col_4: double (nullable = false)
 |-- extra_col_5: double (nullable = false)
 |-- extra_col_6: double (nullable = false)
 |-- extra_col_7: double (nullable = false)
 |-- extra_col_8: double (nullable = false)
 |-- extra_col_9: double (nullable = false)
 |-- extra_col_10: double (nullable = false)
 |-- extra_col_11: double (nullable = false)
 |-- extra_col_12: double (nullable = false)
 |-- extra_col_13: double (nullable = false)

+---+----+-----+----+--------+------------------+--------+------------------+-----------------

---
## 2. CSV vs Parquet — Kích thước file và tốc độ đọc

**Mục tiêu**: thấy được Parquet vừa **nhẹ hơn** (nhờ nén columnar) vừa **đọc nhanh hơn** (nhờ metadata + columnar layout).

In [3]:
# Helper: đo thời gian thực thi một action
def timed(name, func):
    t0 = time.time()
    result = func()
    dt = time.time() - t0
    print(f"  [{name}] {dt:.3f}s  →  result = {result}")
    return dt

# Helper: tính dung lượng thư mục
def dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

In [4]:
# Ghi data ra cả 2 format để so sánh
BASE = "/opt/workspace/data/demo-module-4.4"
csv_path     = f"file://{BASE}/sales_csv"
parquet_path = f"file://{BASE}/sales_parquet"

print("Ghi CSV ...")
df.write.mode("overwrite").option("header", "true").csv(csv_path)

print("Ghi Parquet ...")
df.write.mode("overwrite").parquet(parquet_path)


Ghi CSV ...


Ghi Parquet ...


26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:03:04 WARN MemoryManager: Total allocation exceeds 95.

26/05/15 13:03:15 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
Exception in thread "refresh progress" java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:16 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:17 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:18 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:26 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 24.52% for 31 writers
26/05/15 13:03:27 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:36 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:36 ERROR Utils: Aborting task
java.lang.OutOfMemoryError: Java heap space
26/05/15 13:03:39 ERROR Utils: Aborting task
java

Py4JJavaError: An error occurred while calling o134.parquet.
: org.apache.spark.SparkException: Job 2 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1259)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1257)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1257)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3129)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:84)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:3015)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1375)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:3015)
	at org.apache.spark.SparkContext.$anonfun$stop$12(SparkContext.scala:2258)
	at org.apache.s

In [ ]:
# Đo thời gian count toàn bảng
print("COUNT toàn bảng:")
t_csv = timed("CSV    ", lambda: spark.read.option("header", "true").csv(csv_path).count())
t_pq  = timed("Parquet", lambda: spark.read.parquet(parquet_path).count())
print(f"\n→ Parquet nhanh hơn CSV {t_csv/t_pq:.1f}x lần")

26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 27.14% for 28 writers
26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 28.15% for 27 writers
26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 29.23% for 26 writers
26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 30.40% for 25 writers
26/05/15 13:03:46 WARN FileOutputCommitter: Could not delete file:/opt/workspace/data/demo-module-4.4/sales_parquet/_temporary/0/_temporary/attempt_202605151303031655364247836773619_0002_m_000006_39
26/05/15 13:03:46 WARN FileOutputCommitter: Could not delete file:/opt/workspace/data/demo-module-4.4/sales_parquet/_temporary/0/_temporary/attempt_20260515130303165536424783677

26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 38.00% for 20 writers
26/05/15 13:03:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/05/15 13:03:46 WARN FileOutputCommitter: Could not delete file:/opt/workspace/data/demo-module-4.4/sales_parquet/_temporary/0/_temporary/attempt_202605151303031655364247836773619_0002_m_000013_46
26/05/15 13:03:46 WARN FileOutputCommitter: Could not delete file:/opt/workspace/data/demo-module-4.4/sales_parquet/_temporary/0/_temporary/attempt_202605151303031655364247836773619_0002_m_000030_63
26/05/15 13:03:46 ERROR FileFormatWriter: Job job_202605151303031655364247836773619_0002 aborted.
26/05/15 13:03:46 ERROR FileFormatWriter: Job job_202605151303031655364247836773619_0002 aborted.
26/05/15 13:03:46 ERROR Utils: Uncaught exception in thread Executor task launch worker for ta

26/05/15 13:03:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:03:47 WARN FileOutputCommitter: Could not delete file:/opt/workspace/data/demo-module-4.4/sales_parquet/_temporary/0/_temporary/attempt_202605151303031655364247836773619_0002_m_000005_38
26/05/15 13:03:47 ERROR FileFormatWriter: Job job_202605151303031655364247836773619_0002 aborted.
26/05/15 13:03:47 ERROR Utils: Uncaught exception in thread Executor task launch worker for task 5.0 in stage 2.0 (TID 38)
java.lang.NullPointerException
	at org.apache.spark.scheduler.Task.$anonfun$run$3(Task.scala:146)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1375)
	at org.apache.spark.scheduler.Task.run(Task.scala:144)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$4(Executor.scala:621)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally(SparkErrorUtils.scala:64)
	at org.apache.spark.util.S

**Quan sát**:
- Parquet nén tốt hơn nhiều (~5-10x nhỏ hơn CSV) nhờ tổ chức theo cột + dictionary encoding.
- Khi count, Parquet chỉ cần đọc metadata footer → cực nhanh; CSV phải scan toàn bộ file.

---
## 3. Predicate Pushdown — Filter được đẩy xuống tầng file

**Mục tiêu**: thấy `PushedFilters` trong physical plan và đo tốc độ filter.

In [6]:
# Xem physical plan để kiểm tra PushedFilters
df_pq = spark.read.parquet(parquet_path)
filtered_pq = df_pq.filter((F.col("year") == 2024) & (F.col("city") == "HCM"))

print("="*70)
print("PHYSICAL PLAN (Parquet) — chú ý dòng PushedFilters:")
print("="*70)
filtered_pq.explain(mode="formatted")

PHYSICAL PLAN (Parquet) — chú ý dòng PushedFilters:
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [21]: [id#532L, year#533, month#534, city#535, category#536, amount#537, quantity#538, extra_col_0#539, extra_col_1#540, extra_col_2#541, extra_col_3#542, extra_col_4#543, extra_col_5#544, extra_col_6#545, extra_col_7#546, extra_col_8#547, extra_col_9#548, extra_col_10#549, extra_col_11#550, extra_col_12#551, extra_col_13#552]
Batched: true
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_parquet]
PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(year,2024), EqualTo(city,HCM)]
ReadSchema: struct<id:bigint,year:int,month:int,city:string,category:string,amount:double,quantity:int,extra_col_0:double,extra_col_1:double,extra_col_2:double,extra_col_3:double,extra_col_4:double,extra_col_5:double,extra_col_6:double,extra_col_7:double,extra_col_8:double,extra_col_9:double,extra_col_10:double,extra_col_1

In [7]:
# So sánh với CSV — CSV KHÔNG hỗ trợ pushdown
df_csv = spark.read.option("header", "true").csv(csv_path)
filtered_csv = df_csv.filter((F.col("year") == 2024) & (F.col("city") == "HCM"))

print("="*70)
print("PHYSICAL PLAN (CSV) — KHÔNG có PushedFilters thực sự:")
print("="*70)
filtered_csv.explain(mode="formatted")

PHYSICAL PLAN (CSV) — KHÔNG có PushedFilters thực sự:
== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [21]: [id#592, year#593, month#594, city#595, category#596, amount#597, quantity#598, extra_col_0#599, extra_col_1#600, extra_col_2#601, extra_col_3#602, extra_col_4#603, extra_col_5#604, extra_col_6#605, extra_col_7#606, extra_col_8#607, extra_col_9#608, extra_col_10#609, extra_col_11#610, extra_col_12#611, extra_col_13#612]
Batched: false
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_csv]
PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(city,HCM)]
ReadSchema: struct<id:string,year:string,month:string,city:string,category:string,amount:string,quantity:string,extra_col_0:string,extra_col_1:string,extra_col_2:string,extra_col_3:string,extra_col_4:string,extra_col_5:string,extra_col_6:string,extra_col_7:string,extra_col_8:string,extra_col_9:string,extra_col_10:string,extra_col_11:string,extra_col_12:string,extra_col_13:strin

'file:///opt/workspace/data/demo-module-4.4/sal'

In [8]:
# Đo thời gian filter
print("FILTER year=2024 AND city='HCM':")
t_csv_f = timed("CSV    ", lambda: df_csv.filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
t_pq_f  = timed("Parquet", lambda: df_pq.filter((F.col("year") == 2024) & (F.col("city") == "HCM")).count())
print(f"\n→ Predicate pushdown giúp Parquet nhanh hơn {t_csv_f/t_pq_f:.1f}x lần")

FILTER year=2024 AND city='HCM':
  [CSV    ] 0.533s  →  result = 6
  [Parquet] 0.826s  →  result = 6

→ Predicate pushdown giúp Parquet nhanh hơn 0.6x lần


**Quan sát**: trong physical plan của Parquet, bạn sẽ thấy:
```
PushedFilters: [IsNotNull(year), IsNotNull(city), EqualTo(year,2024), EqualTo(city,HCM)]
```
Đây là **bằng chứng** Spark đã đẩy filter xuống tận tầng file. Với CSV, filter chỉ được áp dụng sau khi đọc xong toàn bộ.

---
## 4. Column Pruning — Chỉ đọc cột cần

**Mục tiêu**: bảng có 20 cột, nhưng query chỉ cần 2 → đo lượng giảm I/O.

In [9]:
# Test với Parquet — column pruning hoạt động
print("PARQUET — đọc 20 cột vs 2 cột:")
t_all_pq = timed("All 20 cols", lambda: spark.read.parquet(parquet_path).select("*").count())
t_2_pq   = timed("2 cols     ", lambda: spark.read.parquet(parquet_path).select("city", "amount").count())

print("\nCSV — column pruning KHÔNG hoạt động (vẫn phải scan cả dòng):")
t_all_csv = timed("All 20 cols", lambda: spark.read.option("header", "true").csv(csv_path).select("*").count())
t_2_csv   = timed("2 cols     ", lambda: spark.read.option("header", "true").csv(csv_path).select("city", "amount").count())

print(f"\n→ Parquet: chọn 2 cột nhanh hơn chọn 20 cột {t_all_pq/t_2_pq:.1f}x lần")
print(f"→ CSV    : chọn 2 cột vs 20 cột tương đương ({t_all_csv/t_2_csv:.1f}x) — không có pruning")

PARQUET — đọc 20 cột vs 2 cột:
  [All 20 cols] 1.590s  →  result = 100
  [2 cols     ] 1.573s  →  result = 100

CSV — column pruning KHÔNG hoạt động (vẫn phải scan cả dòng):
  [All 20 cols] 1.977s  →  result = 100
  [2 cols     ] 1.895s  →  result = 100

→ Parquet: chọn 2 cột nhanh hơn chọn 20 cột 1.0x lần
→ CSV    : chọn 2 cột vs 20 cột tương đương (1.0x) — không có pruning


In [10]:
# Xem physical plan để confirm — chú ý phần "ReadSchema"
print("="*70)
print("ReadSchema chỉ chứa 2 cột được select:")
print("="*70)
spark.read.parquet(parquet_path).select("city", "amount").explain(mode="formatted")

ReadSchema chỉ chứa 2 cột được select:
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [city#1010, amount#1012]
Batched: true
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_parquet]
ReadSchema: struct<city:string,amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [city#1010, amount#1012]




In [11]:
spark.read.option("header", "true").csv(csv_path).select("city", "amount").explain(mode="formatted")

== Physical Plan ==
Scan csv  (1)


(1) Scan csv 
Output [2]: [city#1072, amount#1074]
Batched: false
Location: InMemoryFileIndex [file:/opt/workspace/data/demo-module-4.4/sales_csv]
ReadSchema: struct<city:string,amount:string>




== Optimized Logical Plan ==
Project [city#1505, amount#1507], Statistics(sizeInBytes=13.2 KiB)
+- Relation [id#1502L,year#1503,month#1504,city#1505,category#1506,amount#1507,quantity#1508,extra_col_0#1509,extra_col_1#1510,extra_col_2#1511,extra_col_3#1512,extra_col_4#1513,extra_col_5#1514,extra_col_6#1515,extra_col_7#1516,extra_col_8#1517,extra_col_9#1518,extra_col_10#1519,extra_col_11#1520,extra_col_12#1521,extra_col_13#1522] parquet, Statistics(sizeInBytes=68.8 KiB)

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [city#1505,amount#1507] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:double>





[Stage 49:>                                                         (0 + 1) / 1]



== Optimized Logical Plan ==
Project [city#1567, amount#1569], Statistics(sizeInBytes=3.5 KiB)
+- Relation [id#1564,year#1565,month#1566,city#1567,category#1568,amount#1569,quantity#1570,extra_col_0#1571,extra_col_1#1572,extra_col_2#1573,extra_col_3#1574,extra_col_4#1575,extra_col_5#1576,extra_col_6#1577,extra_col_7#1578,extra_col_8#1579,extra_col_9#1580,extra_col_10#1581,extra_col_11#1582,extra_col_12#1583,extra_col_13#1584] csv, Statistics(sizeInBytes=30.8 KiB)

== Physical Plan ==
FileScan csv [city#1567,amount#1569] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string,amount:string>





---
## 5. Partition Pruning — Bỏ qua thư mục không cần

**Mục tiêu**: ghi data với `partitionBy("year", "month")` và đo lường khi filter trên cột partition.

In [12]:
# Ghi data có partition
partitioned_local = f"{BASE}/sales_partitioned"
partitioned_path = f"file://{partitioned_local}"

print("Ghi data với partitionBy(year, month) ...")
df.write.mode("overwrite").partitionBy("year", "month").parquet(partitioned_path)

# Xem cấu trúc thư mục được tạo:
print("\nCấu trúc thư mục được tạo:")
for root, dirs, files in os.walk(partitioned_local):
    level = root.replace(partitioned_local, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:  # không in file để gọn
        break

Ghi data với partitionBy(year, month) ...


26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:00:29 WARN MemoryManager: Total allocation exceeds 95.


Cấu trúc thư mục được tạo:
sales_partitioned/
  year=2023/
    month=1/


In [13]:
# So sánh: filter trên partitioned vs non-partitioned
print("FILTER year=2024 AND month=6:")

# Không partition — phải scan toàn bộ file
t_no_part = timed("Không partition", 
    lambda: spark.read.parquet(parquet_path)
             .filter((F.col("year") == 2024) & (F.col("month") == 6)).count())

# Có partition — chỉ đọc 1 thư mục
t_part = timed("Có partition   ",
    lambda: spark.read.parquet(partitioned_path)
             .filter((F.col("year") == 2024) & (F.col("month") == 6)).count())

print(f"\n→ Partition pruning nhanh hơn {t_no_part/t_part:.1f}x lần")
print(f"→ Số partition được đọc: 1/36 = {1/36*100:.1f}% dữ liệu")

FILTER year=2024 AND month=6:


  [Không partition] 1.200s  →  result = 0
  [Có partition   ] 4.929s  →  result = 0

→ Partition pruning nhanh hơn 0.2x lần
→ Số partition được đọc: 1/36 = 2.8% dữ liệu


In [14]:
# Xem physical plan — chú ý PartitionFilters
print("="*70)
print("PartitionFilters tách biệt với PushedFilters:")
print("="*70)
(spark.read.parquet(partitioned_path)
  .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
  .explain(False))

PartitionFilters tách biệt với PushedFilters:
== Physical Plan ==
*(1) Filter (isnotnull(city#1274) AND (city#1274 = HCM))
+- *(1) ColumnarToRow
   +- FileScan parquet [id#1273L,city#1274,category#1275,amount#1276,quantity#1277,extra_col_0#1278,extra_col_1#1279,extra_col_2#1280,extra_col_3#1281,extra_col_4#1282,extra_col_5#1283,extra_col_6#1284,extra_col_7#1285,extra_col_8#1286,extra_col_9#1287,extra_col_10#1288,extra_col_11#1289,extra_col_12#1290,extra_col_13#1291,year#1292,month#1293] Batched: true, DataFilters: [isnotnull(city#1274), (city#1274 = HCM)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/demo-module-4.4/sales_partitioned], PartitionFilters: [isnotnull(year#1292), isnotnull(month#1293), (year#1292 = 2024), (month#1293 = 6)], PushedFilters: [IsNotNull(city), EqualTo(city,HCM)], ReadSchema: struct<id:bigint,city:string,category:string,amount:double,quantity:int,extra_col_0:double,extra_...




**Quan sát**: physical plan chia thành 2 nhóm filter:
- `PartitionFilters: [year=2024, month=6]` → Spark chỉ vào đúng thư mục `year=2024/month=6/`.
- `PushedFilters: [EqualTo(city,HCM)]` → trong các file của thư mục đó, Spark còn pushdown thêm filter trên `city` xuống Parquet.

### ⚠️ Demo "small file problem"

Hãy thử partition trên cột high-cardinality để **thấy** vấn đề:

In [15]:
# DEMO: partition theo cột có cardinality cao = THẢM HOẠ
# Ở đây ta partition theo `quantity` (100 giá trị) — đã là quá nhiều với 1M rows

bad_partition_local = f"{BASE}/sales_bad_partition"
bad_partition_path = f"file://{bad_partition_local}"
print("Ghi với partitionBy(quantity) — 100 giá trị khác nhau...")
df.write.mode("overwrite").partitionBy("quantity").parquet(bad_partition_path)

# Đếm số file
n_files = sum(len(files) for _, _, files in os.walk(bad_partition_local))
size_mb = dir_size_mb(bad_partition_local)
print(f"\nSố file tạo ra : {n_files}")
print(f"Dung lượng     : {size_mb:.2f} MB")
print(f"Trung bình/file: {size_mb*1024/n_files:.1f} KB  ← QUÁ NHỎ!")

# So với partition tốt
n_files_good = sum(len(files) for _, _, files in os.walk(partitioned_local))
size_mb_good = dir_size_mb(partitioned_local)
print(f"\nĐể so sánh — partition theo (year, month):")
print(f"Số file        : {n_files_good}")
print(f"Trung bình/file: {size_mb_good*1024/n_files_good:.1f} KB")

Ghi với partitionBy(quantity) — 100 giá trị khác nhau...


26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/05/15 13:01:07 WARN MemoryManager: Total allocation exceeds 95.


Số file tạo ra : 202
Dung lượng     : 0.52 MB
Trung bình/file: 2.6 KB  ← QUÁ NHỎ!

Để so sánh — partition theo (year, month):
Số file        : 202
Trung bình/file: 2.5 KB


**Bài học**: partition sai → hàng trăm/nghìn file nhỏ → đọc chậm, overhead lớn. Quy tắc: **mỗi partition nên có ít nhất 100MB-1GB data**.

---
## 6. Kết hợp cả 3 kỹ thuật

**Mục tiêu**: chạy 1 query thực tế kết hợp partition pruning + column pruning + predicate pushdown.

In [16]:
# Query thực tế: doanh thu thành phố HCM trong tháng 6/2024
print("Query: tổng doanh thu HCM, tháng 6/2024")
print("="*70)

# Cách CHẬM: đọc CSV, không có gì tối ưu
def query_slow():
    return (spark.read.option("header", "true").csv(csv_path)
            .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
            .agg(F.sum(F.col("amount").cast("double")).alias("total"))
            .collect()[0]["total"])

# Cách NHANH: Parquet + partition + column pruning
def query_fast():
    return (spark.read.parquet(partitioned_path)
            .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
            .select("amount")  # chỉ select cột cần
            .agg(F.sum("amount").alias("total"))
            .collect()[0]["total"])

t_slow = timed("CSV (không tối ưu)         ", query_slow)
t_fast = timed("Parquet + partition + pruning", query_fast)

print(f"\n🚀 Tổng tăng tốc: {t_slow/t_fast:.1f}x lần")

Query: tổng doanh thu HCM, tháng 6/2024
  [CSV (không tối ưu)         ] 2.192s  →  result = None
  [Parquet + partition + pruning] 5.101s  →  result = None

🚀 Tổng tăng tốc: 0.4x lần


In [17]:
# Xem plan đầy đủ — cả 3 kỹ thuật cùng hoạt động
print("="*70)
print("PHYSICAL PLAN HOÀN CHỈNH:")
print("="*70)
(spark.read.parquet(partitioned_path)
  .filter((F.col("year") == 2024) & (F.col("month") == 6) & (F.col("city") == "HCM"))
  .select("amount")
  .agg(F.sum("amount"))
  .explain(True))

PHYSICAL PLAN HOÀN CHỈNH:
== Parsed Logical Plan ==
'Aggregate [sum('amount) AS sum(amount)#1527]
+- Project [amount#1484]
   +- Filter (((year#1500 = 2024) AND (month#1501 = 6)) AND (city#1482 = HCM))
      +- Relation [id#1481L,city#1482,category#1483,amount#1484,quantity#1485,extra_col_0#1486,extra_col_1#1487,extra_col_2#1488,extra_col_3#1489,extra_col_4#1490,extra_col_5#1491,extra_col_6#1492,extra_col_7#1493,extra_col_8#1494,extra_col_9#1495,extra_col_10#1496,extra_col_11#1497,extra_col_12#1498,extra_col_13#1499,year#1500,month#1501] parquet

== Analyzed Logical Plan ==
sum(amount): double
Aggregate [sum(amount#1484) AS sum(amount)#1527]
+- Project [amount#1484]
   +- Filter (((year#1500 = 2024) AND (month#1501 = 6)) AND (city#1482 = HCM))
      +- Relation [id#1481L,city#1482,category#1483,amount#1484,quantity#1485,extra_col_0#1486,extra_col_1#1487,extra_col_2#1488,extra_col_3#1489,extra_col_4#1490,extra_col_5#1491,extra_col_6#1492,extra_col_7#1493,extra_col_8#1494,extra_col_9#149

Trong plan, chú ý 3 dòng:
- `PartitionFilters: [...year=2024..., ...month=6...]` → Partition pruning
- `PushedFilters: [...EqualTo(city,HCM)]` → Predicate pushdown
- `ReadSchema: struct<city:string,amount:double>` → Column pruning (chỉ đọc cột cần để filter + tính sum)

---


---
## 8. Tổng kết

| Kỹ thuật | Cần làm gì | Khi nào hiệu quả |
|----------|-----------|------------------|
| **Format Parquet/ORC** | Ghi data bằng `.parquet()` thay vì `.csv()` | Luôn luôn, trừ khi cần human-readable |
| **Predicate Pushdown** | Dùng `.filter()` với điều kiện đơn giản, tránh UDF | Filter trên cột có index/statistics |
| **Column Pruning** | Dùng `.select("col_a", "col_b")` thay vì `select("*")` | Bảng có nhiều cột nhưng query chỉ cần vài cột |
| **Partition Pruning** | `partitionBy()` cột có cardinality thấp (year, region) | Query thường lọc theo cột partition |


### Quy trình tối ưu khi xây pipeline

1. **Format**: Parquet (mặc định) hoặc Delta/Iceberg nếu cần ACID.
2. **Partition**: chọn 1-2 cột cardinality thấp (year/month/region), kiểm tra mỗi partition ≥ 100MB.
3. **Compaction**: gộp file nhỏ định kỳ (tránh small file problem).
4. **Query**: luôn `select()` cụ thể, `filter()` sớm nhất có thể.
5. **Verify**: dùng `.explain(True)` để xác nhận `PartitionFilters`, `PushedFilters`, `ReadSchema` đều đúng.

### Bài tập

1. Tăng `N_ROWS` lên 10 triệu và đo lại — sự khác biệt sẽ rõ ràng hơn nhiều.
2. Thử partition theo `(year, month, city)` — quan sát số thư mục tạo ra.
3. Viết 1 query dùng UDF trong filter — xem trong `explain` filter có còn được pushdown không?
4. Ghi data với các `compression` khác nhau (`snappy`, `gzip`, `zstd`) và so sánh size + read time.

In [18]:
# Cleanup
spark.sql("DROP DATABASE IF EXISTS demo_db CASCADE")
spark.stop()
print("Done!")

Py4JJavaError: An error occurred while calling o29.sql.
: org.apache.spark.SparkException: Unable to create database default as failed to create its directory hdfs://master:9000/tmp/spark-warehouse.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unableToCreateDatabaseAsFailedToCreateDirectoryError(QueryExecutionErrors.scala:540)
	at org.apache.spark.sql.catalyst.catalog.InMemoryCatalog.liftedTree1$1(InMemoryCatalog.scala:126)
	at org.apache.spark.sql.catalyst.catalog.InMemoryCatalog.createDatabase(InMemoryCatalog.scala:119)
	at org.apache.spark.sql.internal.SharedState.externalCatalog$lzycompute(SharedState.scala:159)
	at org.apache.spark.sql.internal.SharedState.externalCatalog(SharedState.scala:140)
	at org.apache.spark.sql.internal.BaseSessionStateBuilder.$anonfun$catalog$1(BaseSessionStateBuilder.scala:154)
	at org.apache.spark.sql.catalyst.catalog.SessionCatalog.externalCatalog$lzycompute(SessionCatalog.scala:123)
	at org.apache.spark.sql.catalyst.catalog.SessionCatalog.ext